# 1. Process Data

Ingests raw GHCN-Daily data and produces a clean, analysis-ready Parquet file.

**Steps:**
1. Load raw CSVs (~933M rows) into Spark with a fixed schema
2. Parse SE Asia station metadata from `ghcnd-stations.txt`
3. Filter to SE Asian stations and valid station IDs
4. Keep only `TMAX`, `TMIN`, `PRCP` elements; drop QA-flagged rows
5. Convert values from tenths to standard units
6. Pivot from long to wide format (one row per station-day)
7. Add date features (`YEAR`, `MONTH`, `DAY_OF_YEAR`, `TEMP_RANGE`)
8. Join with station metadata (country, lat/lon, elevation)
9. Drop rows missing both temperature readings
10. Save to `data/processed/sea_weather_2000_2024.parquet`

**Output:** ~1M station-day rows across 10 SE Asian countries (2000–2024)

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName('GHCN-SEA-Climate') \
    .config('spark.driver.memory', '8g') \
    .config('spark.executor.memory', '4g') \
    .config('spark.sql.adaptive.enabled', 'true') \
    .config('spark.sql.adaptive.skewJoin.enabled', 'true') \
    .config('spark.sql.shuffle.partitions', '200') \
    .config('spark.default.parallelism', '200') \
    .getOrCreate()
    
print(f"Spark version: {spark.version}")

schema = StructType([
    StructField('STATION_ID', StringType()),
    StructField('DATE', StringType()),
    StructField('ELEMENT', StringType()),
    StructField('VALUE', IntegerType()),
    StructField('M_FLAG', StringType()),
    StructField('Q_FLAG', StringType()),
    StructField('S_FLAG', StringType()),
    StructField('OBS_TIME', StringType()),
])

df_raw = spark.read.csv('../data/raw', schema=schema)
print(f"Raw data loaded from ../data/raw/")
df_raw.show(5)

Spark version: 4.1.1
Raw data loaded from ../data/raw/
+-----------+--------+-------+-----+------+------+------+--------+
| STATION_ID|    DATE|ELEMENT|VALUE|M_FLAG|Q_FLAG|S_FLAG|OBS_TIME|
+-----------+--------+-------+-----+------+------+------+--------+
|ASN00066070|20000101|   PRCP|    0|  NULL|  NULL|     a|    NULL|
|ASN00066071|20000101|   PRCP|    0|  NULL|  NULL|     a|    NULL|
|ASN00066072|20000101|   PRCP|    8|  NULL|  NULL|     a|    NULL|
|ASN00066073|20000101|   PRCP|   10|  NULL|  NULL|     a|    NULL|
|ASN00066078|20000101|   PRCP|    0|  NULL|  NULL|     a|    NULL|
+-----------+--------+-------+-----+------+------+------+--------+
only showing top 5 rows


In [2]:
import pandas as pd

stations = pd.read_fwf(
    '../data/raw/ghcnd-stations.txt',
    colspecs=[
        (0, 11),    # ID: columns 1-11
        (12, 20),   # LATITUDE: columns 13-20
        (21, 30),   # LONGITUDE: columns 22-30
        (31, 37),   # ELEVATION: columns 32-37
        (38, 40),   # STATE: columns 39-40
        (41, 71),   # NAME: columns 42-71
        (72, 75),   # GSN_FLAG: columns 73-75
        (76, 79),   # HCN_CRN_FLAG: columns 77-79
        (80, 85),   # WMO_ID: columns 81-85
    ],
    names=['STATION_ID','LATITUDE','LONGITUDE','ELEVATION',
           'STATE','NAME','GSN_FLAG','HCN_CRN_FLAG','WMO_ID'],
    dtype={'STATION_ID': str, 'STATE': str, 'WMO_ID': str}
)

# Extract FIPS country code from first 2 chars of station ID
stations['COUNTRY_CODE'] = stations['STATION_ID'].str[:2]

# Map to country names (from ghcnd-countries.txt)
sea_countries = {
    'VM': 'Vietnam',     'TH': 'Thailand',    'MY': 'Malaysia',
    'ID': 'Indonesia',   'RP': 'Philippines', 'CB': 'Cambodia',
    'LA': 'Laos',        'BM': 'Myanmar',     'SN': 'Singapore',
    'BX': 'Brunei'
}

# Filter to SE Asia stations only
stations_sea = stations[
    stations['COUNTRY_CODE'].isin(sea_countries.keys())
].copy()
stations_sea['COUNTRY'] = stations_sea['COUNTRY_CODE'].map(sea_countries)

# Handle missing elevation (-999.9 = missing)
stations_sea.loc[stations_sea['ELEVATION'] == -999.9, 'ELEVATION'] = None

print(f'SE Asia stations: {len(stations_sea)}')  # Expected: 287
print(stations_sea['COUNTRY'].value_counts())


SE Asia stations: 287
COUNTRY
Indonesia      104
Vietnam         56
Thailand        54
Philippines     34
Malaysia        16
Laos            15
Myanmar          5
Brunei           1
Cambodia         1
Singapore        1
Name: count, dtype: int64


In [3]:
# Convert stations_sea (Pandas) to Spark DataFrame for joining later

from pyspark.sql.functions import col, when, to_date, year, month, dayofyear, first, count

stations_spark = spark.createDataFrame(stations_sea)
print(f"Stations ready: {len(stations_sea)} SE Asia stations")

Stations ready: 287 SE Asia stations


In [4]:
# Filter df_raw to SE Asian stations only

sea_prefixes = ['VM','TH','MY','ID','RP','CB','LA','BM','SN','BX']

df_sea = df_raw.filter(
    df_raw.STATION_ID.substr(1, 2).isin(sea_prefixes)
)

# Ensure STATION_ID is valid (2 letters + 9 alphanumeric = 11 chars)
from pyspark.sql.functions import length, countDistinct

df_sea = df_sea.filter(
    (length(col('STATION_ID')) == 11) &
    (col('STATION_ID').rlike('^[A-Z]{2}[A-Z0-9]{9}$'))
)

# Cache early to avoid recomputation
df_sea.cache()

# Get all stats in one action
stats = df_sea.agg(
    count('*').alias('total_rows'),
    countDistinct('STATION_ID').alias('unique_stations')
).collect()[0]

print(f"Filtered to SE Asia: {stats.total_rows:,} rows")
print(f"Unique stations: {stats.unique_stations}")
print(f"Matches metadata: {stats.unique_stations} stations (expected ~219)")

Filtered to SE Asia: 5,115,078 rows
Unique stations: 219
Matches metadata: 219 stations (expected ~219)


In [5]:
# Filter to key elements only (TMAX, TMIN, PRCP)
# Note: AWND (wind) is not available in SE Asia GHCN data

df_filtered = df_sea.filter(
    df_sea.ELEMENT.isin(['TMAX', 'TMIN', 'PRCP'])
)
print(f"Filtered to key elements: {df_filtered.count():,} rows")
print(f"Element breakdown:")
df_filtered.groupBy('ELEMENT').count().orderBy('ELEMENT').show()

Filtered to key elements: 3,313,251 rows
Element breakdown:
+-------+-------+
|ELEMENT|  count|
+-------+-------+
|   PRCP| 646762|
|   TMAX|1400102|
|   TMIN|1266387|
+-------+-------+



In [6]:
# Quality flag filtering
# Q_FLAG blank/null = passed all QA checks


df_clean = df_filtered.filter(
    (df_filtered.Q_FLAG.isNull()) | (df_filtered.Q_FLAG == '')
)
removed = df_filtered.count() - df_clean.count()
print(f"Quality filtered: {df_clean.count():,} rows")
print(f"Removed {removed:,} flagged observations")

Quality filtered: 3,312,234 rows
Removed 1,017 flagged observations


In [7]:
# Unit conversion: all values stored as tenths, divide by 10

df_converted = df_clean.withColumn(
    'VALUE',
    col('VALUE').cast('double') / 10.0
)
print(f"Unit conversion complete (all VALUES divided by 10)")

Unit conversion complete (all VALUES divided by 10)


In [8]:
# Pivot ELEMENT column from long format to wide format

# Pre-repartition for better pivot performance
df_to_pivot = df_converted.repartition('STATION_ID', 'DATE')

df_pivot = df_to_pivot.groupBy('STATION_ID', 'DATE') \
    .pivot('ELEMENT', ['TMAX', 'TMIN', 'PRCP']) \
    .agg(first('VALUE'))

# Cache after pivot
df_pivot.cache()

print("Pivoted to wide format - showing sample")
df_pivot.show(5)

Pivoted to wide format - showing sample
+-----------+--------+----+----+----+
| STATION_ID|    DATE|TMAX|TMIN|PRCP|
+-----------+--------+----+----+----+
|IDM00096685|20000101|30.2|23.4|NULL|
|RPM00098233|20000102|NULL|20.0| 0.0|
|VMM00048848|20000102|23.0|18.0| 0.0|
|TH000048376|20000103|32.7|18.2|NULL|
|VMM00048840|20000103|23.6|NULL| 0.0|
+-----------+--------+----+----+----+
only showing top 5 rows


In [9]:
# Date parsing & feature engineering

df_featured = df_pivot \
    .withColumn('DATE', to_date(col('DATE'), 'yyyyMMdd')) \
    .withColumn('YEAR', year(col('DATE'))) \
    .withColumn('MONTH', month(col('DATE'))) \
    .withColumn('DAY_OF_YEAR', dayofyear(col('DATE'))) \
    .withColumn('TEMP_RANGE', col('TMAX') - col('TMIN'))   # daily temp swing

print(f"Date parsed and features added: YEAR, MONTH, DAY_OF_YEAR, TEMP_RANGE")
df_featured.printSchema()

Date parsed and features added: YEAR, MONTH, DAY_OF_YEAR, TEMP_RANGE
root
 |-- STATION_ID: string (nullable = true)
 |-- DATE: date (nullable = true)
 |-- TMAX: double (nullable = true)
 |-- TMIN: double (nullable = true)
 |-- PRCP: double (nullable = true)
 |-- YEAR: integer (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- DAY_OF_YEAR: integer (nullable = true)
 |-- TEMP_RANGE: double (nullable = true)



In [10]:
# Join with station metadata

from pyspark.sql.functions import broadcast

# Select needed columns early (reduce data)
stations_small = stations_spark.select(
    'STATION_ID', 'LATITUDE', 'LONGITUDE',
    'ELEVATION', 'COUNTRY', 'COUNTRY_CODE', 'NAME'
)

# Broadcast join (stations_small is tiny ~287 rows)
df_joined = df_featured.join(
    broadcast(stations_small),
    on='STATION_ID',
    how='left'
)

print(f"Joined with station metadata (broadcast join)")
df_joined.select('STATION_ID', 'DATE', 'TMAX', 'TMIN', 'PRCP', 'COUNTRY', 'LATITUDE', 'LONGITUDE').show(5)

Joined with station metadata (broadcast join)
+-----------+----------+----+----+----+-----------+--------+---------+
| STATION_ID|      DATE|TMAX|TMIN|PRCP|    COUNTRY|LATITUDE|LONGITUDE|
+-----------+----------+----+----+----+-----------+--------+---------+
|IDM00096685|2000-01-01|30.2|23.4|NULL|  Indonesia|  -3.442|  114.763|
|RPM00098233|2000-01-02|NULL|20.0| 0.0|Philippines|  17.638|  121.731|
|VMM00048848|2000-01-02|23.0|18.0| 0.0|    Vietnam|  17.483|    106.6|
|TH000048376|2000-01-03|32.7|18.2|NULL|   Thailand|  16.883|    99.15|
|VMM00048840|2000-01-03|23.6|NULL| 0.0|    Vietnam|   19.75|  105.783|
+-----------+----------+----+----+----+-----------+--------+---------+
only showing top 5 rows


In [11]:
# Null rates before final temperature filter
from pyspark.sql.functions import count, when

_nulls = df_joined.agg(
    count('*').alias('total'),
    count(when(col('TMAX').isNull(), 1)).alias('null_tmax'),
    count(when(col('TMIN').isNull(), 1)).alias('null_tmin'),
    count(when(col('PRCP').isNull(), 1)).alias('null_prcp'),
).collect()[0]

print('Null rates after pivot, before temperature filter:')
print(f'  TMAX missing: {_nulls.null_tmax:>7,} / {_nulls.total:,}  ({100*_nulls.null_tmax/_nulls.total:.1f}%)')
print(f'  TMIN missing: {_nulls.null_tmin:>7,} / {_nulls.total:,}  ({100*_nulls.null_tmin/_nulls.total:.1f}%)')
print(f'  PRCP missing: {_nulls.null_prcp:>7,} / {_nulls.total:,}  ({100*_nulls.null_prcp/_nulls.total:.1f}%)')
print(f'  (PRCP nulls are retained — null means no precipitation record, not zero rain)')


Null rates after pivot, before temperature filter:
  TMAX missing: 311,900 / 1,711,511  (18.2%)
  TMIN missing: 445,639 / 1,711,511  (26.0%)
  PRCP missing: 1,064,760 / 1,711,511  (62.2%)
  (PRCP nulls are retained — null means no precipitation record, not zero rain)


In [12]:
df_final = df_joined.filter(
    col('TMAX').isNotNull() & col('TMIN').isNotNull()
)

print(f"Filtered to {df_final.count():,} rows (dropped missing TMAX or TMIN)")

Filtered to 1,011,376 rows (dropped missing TMAX or TMIN)


In [13]:
# Save as single Parquet file

df_final.write.mode("overwrite").parquet("../data/processed/sea_weather_2000_2024.parquet")
print("Saved → data/processed/sea_weather_2000_2024.parquet")

Saved → data/processed/sea_weather_2000_2024.parquet


In [14]:
from pyspark.sql.functions import count, countDistinct

df_check = spark.read.parquet("../data/processed/sea_weather_2000_2024.parquet")

stats = df_check.agg(
    count('*').alias('total_rows'),
    countDistinct('STATION_ID').alias('unique_stations'),
    countDistinct('COUNTRY').alias('unique_countries')
).collect()[0]

print("FINAL DATASET SUMMARY")
print(f"Rows:               {stats.total_rows:,}")
print(f"Unique stations:    {stats.unique_stations}")
print(f"Countries:          {stats.unique_countries}")
print(f"Columns:            {len(df_check.columns)}")

print("\nCountry distribution:")
df_check.groupBy('COUNTRY').count().orderBy('count', ascending=False).show()

print("\nSample rows:")
df_check.select('STATION_ID','DATE','YEAR','MONTH','TMAX','TMIN','TEMP_RANGE','PRCP','COUNTRY').show(5)

FINAL DATASET SUMMARY
Rows:               1,011,376
Unique stations:    218
Countries:          10
Columns:            15

Country distribution:
+-----------+------+
|    COUNTRY| count|
+-----------+------+
|  Indonesia|490634|
|   Thailand|256662|
|Philippines|109166|
|    Vietnam| 55192|
|   Malaysia| 47226|
|    Myanmar| 23737|
|       Laos| 20582|
|     Brunei|  4888|
|  Singapore|  3164|
|   Cambodia|   125|
+-----------+------+


Sample rows:
+-----------+----------+----+-----+----+----+------------------+----+---------+
| STATION_ID|      DATE|YEAR|MONTH|TMAX|TMIN|        TEMP_RANGE|PRCP|  COUNTRY|
+-----------+----------+----+-----+----+----+------------------+----+---------+
|ID000096163|2000-01-01|2000|    1|30.8|23.6| 7.199999999999999|NULL|Indonesia|
|IDM00097760|2000-01-02|2000|    1|31.2|25.2|               6.0|NULL|Indonesia|
|TH000048375|2000-01-03|2000|    1|33.0|16.5|              16.5| 0.0| Thailand|
|TH000048379|2000-01-04|2000|    1|34.0|18.4|15.600000000000001| 0